# DWAV Assignment — Highest-Grossing Films

This notebook parses the Wikipedia page **List of highest-grossing films**, enriches the data using each film's Wikipedia page, stores the result in SQLite, and exports the final dataset to JSON for the static website.

**Final fields**
- title
- release_year
- director
- box_office
- country
- poster_url

## 1. Import libraries and configure paths

In [8]:
from pathlib import Path
from urllib.parse import urljoin
import json
import re
import sqlite3
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

LIST_URL = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"
BASE_URL = "https://en.wikipedia.org"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; DWAVAssignment/1.0)"
}

PROJECT_ROOT = Path(r"C:\Users\belelvser\PycharmProjects\assignment_dwav")

DATA_DIR = PROJECT_ROOT / "data"
DOCS_DIR = PROJECT_ROOT / "docs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "films.db"
JSON_PATH = DATA_DIR / "films.json"
WEB_JSON_PATH = DOCS_DIR / "films.json"

print("Project root:", PROJECT_ROOT)
print("Database path:", DB_PATH)
print("JSON path:", JSON_PATH)
print("Web JSON path:", WEB_JSON_PATH)

session = requests.Session()
session.headers.update(HEADERS)

Project root: C:\Users\belelvser\PycharmProjects\assignment_dwav
Database path: C:\Users\belelvser\PycharmProjects\assignment_dwav\data\films.db
JSON path: C:\Users\belelvser\PycharmProjects\assignment_dwav\data\films.json
Web JSON path: C:\Users\belelvser\PycharmProjects\assignment_dwav\docs\films.json


## 2. Helper functions

In [2]:
def get_html(url: str) -> str:
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response.text


def clean_text(text):
    if text is None:
        return None

    if isinstance(text, bytes):
        text = text.decode("utf-8", errors="replace")

    text = str(text)
    text = text.replace("\xa0", " ").replace("\u200b", "").replace("\ufeff", "")
    text = re.sub(r"\[[^\]]*\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s*[†‡*]+\s*$", "", text)
    text = text.strip(" ;,\n\t")
    return text if text else None


def normalize_header(text):
    text = clean_text(text) or ""
    return re.sub(r"[^a-z0-9]+", "", text.lower())


def parse_money_to_int(value):
    if value is None:
        return None

    text = clean_text(value)
    if not text:
        return None

    lowered = text.lower()
    text = re.split(r"[;(]", text)[0].strip()
    text_no_commas = text.replace(",", "")

    if "billion" in lowered:
        match = re.search(r"(\d+(?:\.\d+)?)", text_no_commas)
        return int(float(match.group(1)) * 1_000_000_000) if match else None

    if "million" in lowered:
        match = re.search(r"(\d+(?:\.\d+)?)", text_no_commas)
        return int(float(match.group(1)) * 1_000_000) if match else None

    match = re.search(r"(\d{1,3}(?:,\d{3})+|\d{7,})", text)
    return int(match.group(1).replace(",", "")) if match else None


def extract_year_from_release_date(value):
    text = clean_text(value)
    if not text:
        return None

    years = re.findall(r"\b(19\d{2}|20\d{2})\b", text)
    return int(years[0]) if years else None


def extract_year_fallback(film_soup: BeautifulSoup, film_url: str = ""):
    candidates = []

    heading = film_soup.find("h1")
    if heading:
        candidates.append(heading.get_text(" ", strip=True))

    if film_soup.title:
        candidates.append(film_soup.title.get_text(" ", strip=True))

    if film_url:
        candidates.append(film_url)

    for text in candidates:
        years = re.findall(r"\b(19\d{2}|20\d{2})\b", str(text))
        if years:
            return int(years[0])

    return None

## 3. Parse the main Wikipedia table

In [3]:
def find_main_table(page_soup: BeautifulSoup):
    tables = page_soup.find_all("table", class_=lambda c: c and "wikitable" in c)

    for table in tables:
        for tr in table.find_all("tr"):
            header_cells = tr.find_all("th", recursive=False)
            if not header_cells:
                continue

            headers = [clean_text(th.get_text(" ", strip=True)) for th in header_cells]
            normalized = [normalize_header(h) for h in headers if h]

            if "title" in normalized and "year" in normalized and any("gross" in h for h in normalized):
                return table

    raise ValueError("Main highest-grossing films table was not found.")


def get_column_indices(table):
    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"], recursive=False)
        if not cells:
            continue

        headers = [normalize_header(cell.get_text(" ", strip=True)) for cell in cells]
        if "title" in headers and "year" in headers and any("gross" in h for h in headers):
            title_idx = headers.index("title")
            year_idx = headers.index("year")
            gross_idx = next(i for i, h in enumerate(headers) if "gross" in h)
            return {"title": title_idx, "gross": gross_idx, "year": year_idx}

    raise ValueError("Required columns were not found.")


def extract_film_link(title_cell):
    links = [
        a for a in title_cell.find_all("a", href=True)
        if a["href"].startswith("/wiki/") and "cite_note" not in a["href"]
    ]
    return urljoin(BASE_URL, links[0]["href"]) if links else None


def extract_main_rows(table):
    rows = []
    col = get_column_indices(table)

    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"], recursive=False)
        if not cells:
            continue

        normalized_cells = [normalize_header(cell.get_text(" ", strip=True)) for cell in cells]
        if "title" in normalized_cells and "year" in normalized_cells:
            continue

        max_idx = max(col["title"], col["gross"], col["year"])
        if len(cells) <= max_idx:
            continue

        title_cell = cells[col["title"]]
        gross_cell = cells[col["gross"]]
        year_cell = cells[col["year"]]

        for sup in title_cell.find_all("sup"):
            sup.decompose()

        title = clean_text(title_cell.get_text(" ", strip=True))
        year_text = clean_text(year_cell.get_text(" ", strip=True))
        gross_text = clean_text(gross_cell.get_text(" ", strip=True))

        if not title:
            continue

        year_match = re.search(r"(19|20)\d{2}", year_text or "")
        release_year = int(year_match.group(0)) if year_match else None

        rows.append({
            "title": title,
            "release_year": release_year,
            "box_office": parse_money_to_int(gross_text),
            "film_url": extract_film_link(title_cell)
        })

    return pd.DataFrame(rows).drop_duplicates(subset=["title", "release_year"]).reset_index(drop=True)


main_html = get_html(LIST_URL)
soup = BeautifulSoup(main_html, "lxml")
main_table = find_main_table(soup)
base_df = extract_main_rows(main_table)

display(base_df.head())
print("Films extracted from the main table:", len(base_df))

,title,release_year,box_office,film_url
0,Avatar,2009,2923710708,https://en.wikipedia.org/wiki/Avatar_(2009_film)
1,Avengers: Endgame,2019,2797501328,https://en.wikipedia.org/wiki/Avengers:_Endgame
2,Avatar: The Way of Water,2022,2334484620,https://en.wikipedia.org/wiki/Avatar:_The_Way_...
3,Titanic,1997,2257906828,https://en.wikipedia.org/wiki/Titanic_(1997_film)
4,Ne Zha 2,2025,2215690000,https://en.wikipedia.org/wiki/Ne_Zha_2


Films extracted from the main table: 50


## 4. Enrich each film with director, country, and poster URL

In [4]:
def cell_to_clean_string(cell):
    if cell is None:
        return None

    for sup in cell.find_all("sup"):
        sup.decompose()

    list_items = [clean_text(li.get_text(" ", strip=True)) for li in cell.find_all("li")]
    list_items = [x for x in list_items if x]
    if list_items:
        return ", ".join(list_items)

    link_texts = [clean_text(a.get_text(" ", strip=True)) for a in cell.find_all("a")]
    link_texts = [x for x in link_texts if x]
    if link_texts:
        unique = []
        for item in link_texts:
            if item not in unique:
                unique.append(item)
        return ", ".join(unique)

    return clean_text(cell.get_text(" ", strip=True))


def find_infobox_value(film_soup: BeautifulSoup, labels):
    infobox = film_soup.find("table", class_=lambda c: c and "infobox" in c)
    if not infobox:
        return None

    normalized_labels = [normalize_header(label) for label in labels]

    for row in infobox.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue

        header = normalize_header(th.get_text(" ", strip=True))
        if header in normalized_labels:
            return cell_to_clean_string(td)

    return None


def extract_poster_url(film_soup: BeautifulSoup):
    infobox = film_soup.find("table", class_=lambda c: c and "infobox" in c)
    if not infobox:
        return None

    img = infobox.find("img")
    if not img:
        return None

    src = img.get("src")
    if not src:
        return None

    if src.startswith("//"):
        return "https:" + src
    if src.startswith("/"):
        return BASE_URL + src
    return src


def extract_film_details(film_url: str):
    if not film_url:
        return {
            "director": None,
            "country": None,
            "page_year": None,
            "poster_url": None
        }

    try:
        html = get_html(film_url)
        film_soup = BeautifulSoup(html, "lxml")

        director = find_infobox_value(film_soup, ["Directed by"])
        country = find_infobox_value(film_soup, ["Country", "Countries"])
        release_date = find_infobox_value(film_soup, ["Release date", "Release dates"])

        page_year = extract_year_from_release_date(release_date)
        if page_year is None:
            page_year = extract_year_fallback(film_soup, film_url)

        return {
            "director": clean_text(director),
            "country": clean_text(country),
            "page_year": page_year,
            "poster_url": extract_poster_url(film_soup)
        }

    except Exception:
        return {
            "director": None,
            "country": None,
            "page_year": None,
            "poster_url": None
        }


details = []
for i, url in enumerate(base_df["film_url"], start=1):
    details.append(extract_film_details(url))
    if i % 10 == 0 or i == len(base_df):
        print(f"Processed {i} / {len(base_df)} films")
    time.sleep(0.35)

details_df = pd.DataFrame(details)
display(details_df.head())

Processed 10 / 50 films
Processed 20 / 50 films
Processed 30 / 50 films
Processed 40 / 50 films
Processed 50 / 50 films


,director,country,page_year,poster_url
0,James Cameron,"United States, United Kingdom",2009,https://upload.wikimedia.org/wikipedia/en/d/d6...
1,"Anthony Russo, Joe Russo",United States,2019,https://upload.wikimedia.org/wikipedia/en/0/0d...
2,James Cameron,United States,2022,https://upload.wikimedia.org/wikipedia/en/5/54...
3,James Cameron,United States,1997,https://upload.wikimedia.org/wikipedia/en/thum...
4,Jiaozi,China,2025,https://upload.wikimedia.org/wikipedia/en/thum...


## 5. Build the final dataset

In [5]:
films_df = pd.concat([base_df, details_df], axis=1)
films_df["release_year"] = films_df["release_year"].fillna(films_df["page_year"])

for column in ["title", "director", "country"]:
    films_df[column] = films_df[column].apply(clean_text)

if "poster_url" not in films_df.columns:
    films_df["poster_url"] = None

films_df = films_df[["title", "release_year", "director", "box_office", "country", "poster_url"]]
films_df = films_df.drop_duplicates(subset=["title"]).reset_index(drop=True)

films_df["release_year"] = pd.to_numeric(films_df["release_year"], errors="coerce").astype("Int64")
films_df["box_office"] = pd.to_numeric(films_df["box_office"], errors="coerce")

display(films_df.head())
print("Final dataset shape:", films_df.shape)

,title,release_year,director,box_office,country,poster_url
0,Avatar,2009,James Cameron,2923710708,"United States, United Kingdom",https://upload.wikimedia.org/wikipedia/en/d/d6...
1,Avengers: Endgame,2019,"Anthony Russo, Joe Russo",2797501328,United States,https://upload.wikimedia.org/wikipedia/en/0/0d...
2,Avatar: The Way of Water,2022,James Cameron,2334484620,United States,https://upload.wikimedia.org/wikipedia/en/5/54...
3,Titanic,1997,James Cameron,2257906828,United States,https://upload.wikimedia.org/wikipedia/en/thum...
4,Ne Zha 2,2025,Jiaozi,2215690000,China,https://upload.wikimedia.org/wikipedia/en/thum...


Final dataset shape: (50, 6)


## 6. Store the data in SQLite

In [6]:
def to_python_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return value
    return value


connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()

cursor.execute("DROP TABLE IF EXISTS films")
cursor.execute("""
CREATE TABLE films (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    release_year INTEGER,
    director TEXT,
    box_office REAL,
    country TEXT,
    poster_url TEXT
)
""")

records = [
    tuple(to_python_value(v) for v in row)
    for row in films_df[["title", "release_year", "director", "box_office", "country", "poster_url"]]
    .itertuples(index=False, name=None)
]

cursor.executemany(
    """
    INSERT INTO films (title, release_year, director, box_office, country, poster_url)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    records
)
connection.commit()

preview_df = pd.read_sql_query(
    """
    SELECT *
    FROM films
    ORDER BY box_office DESC
    LIMIT 10
    """,
    connection
)
display(preview_df)

,id,title,release_year,director,box_office,country,poster_url
0,1,Avatar,2009,James Cameron,2.923711e+09,"United States, United Kingdom",https://upload.wikimedia.org/wikipedia/en/d/d6...
1,2,Avengers: Endgame,2019,"Anthony Russo, Joe Russo",2.797501e+09,United States,https://upload.wikimedia.org/wikipedia/en/0/0d...
2,3,Avatar: The Way of Water,2022,James Cameron,2.334485e+09,United States,https://upload.wikimedia.org/wikipedia/en/5/54...
3,4,Titanic,1997,James Cameron,2.257907e+09,United States,https://upload.wikimedia.org/wikipedia/en/thum...
4,5,Ne Zha 2,2025,Jiaozi,2.215690e+09,China,https://upload.wikimedia.org/wikipedia/en/thum...
5,6,Star Wars: The Force Awakens,2015,J. J. Abrams,2.068224e+09,United States,https://upload.wikimedia.org/wikipedia/en/a/a2...
6,7,Avengers: Infinity War,2018,"Anthony Russo, Joe Russo",2.048360e+09,United States,https://upload.wikimedia.org/wikipedia/en/4/4d...
7,8,Spider-Man: No Way Home,2021,Jon Watts,1.922599e+09,United States,https://upload.wikimedia.org/wikipedia/en/thum...
8,9,Zootopia 2,2025,"Jared Bush, Byron Howard",1.866641e+09,United States,https://upload.wikimedia.org/wikipedia/en/thum...
9,10,Inside Out 2,2024,Kelsey Mann,1.698864e+09,United States,https://upload.wikimedia.org/wikipedia/en/thum...


## 7. Export to JSON for the website

In [9]:
connection.close()

export_conn = sqlite3.connect(DB_PATH)
export_df = pd.read_sql_query(
    """
    SELECT title, release_year, director, box_office, country, poster_url
    FROM films
    ORDER BY box_office DESC
    """,
    export_conn
)
export_conn.close()

records = export_df.to_dict(orient="records")

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

with open(WEB_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("Saved:", JSON_PATH)
print("Saved:", WEB_JSON_PATH)
display(export_df.head(3))

Saved: C:\Users\belelvser\PycharmProjects\assignment_dwav\data\films.json
Saved: C:\Users\belelvser\PycharmProjects\assignment_dwav\docs\films.json


,title,release_year,director,box_office,country,poster_url
0,Avatar,2009,James Cameron,2.923711e+09,"United States, United Kingdom",https://upload.wikimedia.org/wikipedia/en/d/d6...
1,Avengers: Endgame,2019,"Anthony Russo, Joe Russo",2.797501e+09,United States,https://upload.wikimedia.org/wikipedia/en/0/0d...
2,Avatar: The Way of Water,2022,James Cameron,2.334485e+09,United States,https://upload.wikimedia.org/wikipedia/en/5/54...


## 8. Conclusion

The notebook:
1. parsed the main Wikipedia table of highest-grossing films,
2. enriched the data using each film's page,
3. stored the cleaned dataset in SQLite,
4. exported the final records to JSON for the website.